# Global Configuration

In [20]:
# 1. Define your feature groups
identifiers = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Label', 'Attack']
network_features = ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration']
context_features = ['Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Flow Byts/s', 'Flow Pkts/s', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Fwd Act Data Pkts', 'Down/Up Ratio', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
knowledge_features = ['Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Seg Size Min', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var']

# Specify the features to keep
features_to_keep = [
    #selected features
]
# 2. SELECT CONFIGURATION HERE
CONFIG = 'A'  # Change to 'A', 'B', 'C', or 'D'

# 3. Apply Configuration rules dynamically
if CONFIG == 'A':
    features_to_keep = identifiers + network_features
    active_edges = {'network'}
elif CONFIG == 'B':
    features_to_keep = identifiers + network_features + context_features
    active_edges = {'network', 'context'}
elif CONFIG == 'C':
    features_to_keep = identifiers + network_features + knowledge_features
    active_edges = {'network', 'knowledge'}
elif CONFIG == 'D':
    features_to_keep = identifiers + network_features + context_features + knowledge_features
    active_edges = {'network', 'context', 'knowledge'}

# Create the train datastracture for 3 edges

In [21]:
import pandas as pd

# Load your dataset
data = pd.read_csv('train_data.csv')  # Replace 'your_data.csv' with your actual file name

# Keep only the specified features
filtered_data = data[features_to_keep].copy()

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_train_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.")


/var/folders/vw/nm4pm2012rj1j5k7yc6x23bw0000gn/T/ipykernel_88320/3102713459.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])


Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.


In [22]:
#check for inside of csv (just for test, no need for run)
import pandas as pd

# Load the CSV file
file_path = "filtered_train_3edge.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Check if the label column contains '1'
label_column = 'Label'  # Replace with the actual label column name if different
if label_column in df.columns:
    label_distribution = df[label_column].value_counts()
    print("Label Distribution:")
    print(label_distribution)

    if 1 in label_distribution.index:
        print("The CSV contains label '1'.")
    else:
        print("The CSV does NOT contain label '1'.")
else:
    print(f"'{label_column}' column not found in the CSV.")

Label Distribution:
Label
1    366437
0    325040
Name: count, dtype: int64
The CSV contains label '1'.


# created hourly graph with 3 edges from train dataset

In [23]:
import pandas as pd
import networkx as nx
import os

def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Group the DataFrame into hourly slices using the datetime index.
    # (Assumes the DataFrame index is already a DateTimeIndex)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]
    
    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())
        
        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']
            
            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges conditionally based on the active config
            # 1. Network Edge
            if 'network' in active_edges:
                net_attrs = {feat: row[feat] for feat in network_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='network', label=label, interaction='network_communication', **net_attrs)

            # 2. Context Edge
            if 'context' in active_edges:
                ctx_attrs = {feat: row[feat] for feat in context_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='context', label=label, interaction='context', **ctx_attrs)

            # 3. Knowledge Edge
            if 'knowledge' in active_edges:
                knw_attrs = {feat: row[feat] for feat in knowledge_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='knowledge', label=label, interaction='knowledge', **knw_attrs)

        # Save the graph as a .gpickle file.
        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        nx.write_gpickle(G, graph_path)
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_train_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_trai_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)


Hour 0:
Label
0    6
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_trai_h_graphs/test_graph_hour_0.gpickle
Hour 1:
Label
0    10
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_trai_h_graphs/test_graph_hour_1.gpickle
Hour 2:
Label
0    16
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_trai_h_graphs/test_graph_hour_2.gpickle
Hour 3:
Label
0    3
Name: count, dtype: int64
Test graph for hour 3 saved to 3ed_trai_h_graphs/test_graph_hour_3.gpickle
Hour 652:
Label
0    11
Name: count, dtype: int64
Test graph for hour 652 saved to 3ed_trai_h_graphs/test_graph_hour_652.gpickle
Hour 653:
Label
0    15
Name: count, dtype: int64
Test graph for hour 653 saved to 3ed_trai_h_graphs/test_graph_hour_653.gpickle
Hour 654:
Label
0    18
Name: count, dtype: int64
Test graph for hour 654 saved to 3ed_trai_h_graphs/test_graph_hour_654.gpickle
Hour 655:
Label
0    5
Name: count, dtype: int64
Test graph for hour 655 saved to 3ed_trai_h_graphs/test_graph_hour_655.gpi

In [24]:
#########Riplika of previous code:
import pandas as pd
import networkx as nx
import os

def add_node_features(G):
    """
    Adds additional features to nodes in the graph, including:
    - Node degree
    - Community ID
    - Temporal activity (average edge count per node)
    - Node centrality (betweenness centrality)

    Parameters:
        G (nx.MultiDiGraph): The input graph.

    Returns:
        nx.MultiDiGraph: The graph with added node features.
    """
    # Add degree
    for node in G.nodes:
        G.nodes[node]['degree'] = G.degree[node]

    # Add community detection (Label Propagation)
    undirected_graph = nx.Graph(G)  # Convert to undirected for community detection
    communities = nx.community.label_propagation_communities(undirected_graph)
    community_mapping = {node: community_id for community_id, community in enumerate(communities) for node in community}
    for node in G.nodes:
        G.nodes[node]['community'] = community_mapping.get(node, -1)

    # Add centrality (Betweenness Centrality)
    centrality = nx.betweenness_centrality(G)
    for node, value in centrality.items():
        G.nodes[node]['centrality'] = value

    return G

def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Group the DataFrame into hourly slices using the datetime index.
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]
    
    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())
        
        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']
            
            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges conditionally based on the active config
            # 1. Network Edge
            if 'network' in active_edges:
                net_attrs = {feat: row[feat] for feat in network_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='network', label=label, interaction='network_communication', **net_attrs)

            # 2. Context Edge
            if 'context' in active_edges:
                ctx_attrs = {feat: row[feat] for feat in context_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='context', label=label, interaction='context', **ctx_attrs)

            # 3. Knowledge Edge
            if 'knowledge' in active_edges:
                knw_attrs = {feat: row[feat] for feat in knowledge_features if feat in row}
                G.add_edge(src_ip, dst_ip, key='knowledge', label=label, interaction='knowledge', **knw_attrs)

        # Add node features
        G = add_node_features(G)

        # Save the graph as a .gpickle file.
        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        nx.write_gpickle(G, graph_path)
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_train_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_trai_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)


Hour 0:
Label
0    6
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_trai_h_graphs/test_graph_hour_0.gpickle
Hour 1:
Label
0    10
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_trai_h_graphs/test_graph_hour_1.gpickle
Hour 2:
Label
0    16
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_trai_h_graphs/test_graph_hour_2.gpickle
Hour 3:
Label
0    3
Name: count, dtype: int64
Test graph for hour 3 saved to 3ed_trai_h_graphs/test_graph_hour_3.gpickle
Hour 652:
Label
0    11
Name: count, dtype: int64
Test graph for hour 652 saved to 3ed_trai_h_graphs/test_graph_hour_652.gpickle
Hour 653:
Label
0    15
Name: count, dtype: int64
Test graph for hour 653 saved to 3ed_trai_h_graphs/test_graph_hour_653.gpickle
Hour 654:
Label
0    18
Name: count, dtype: int64
Test graph for hour 654 saved to 3ed_trai_h_graphs/test_graph_hour_654.gpickle
Hour 655:
Label
0    5
Name: count, dtype: int64
Test graph for hour 655 saved to 3ed_trai_h_graphs/test_graph_hour_655.gpi

# Community detection for graphs and then update the graph with the label of community for each node

In [25]:
import networkx as nx
import os

def detect_and_label_communities_lpa(graph):
    """
    Perform community detection using the Label Propagation Algorithm (LPA) and label nodes with community IDs.
    Adds 'x' attribute based on the 'community' label.

    Parameters:
        graph (nx.MultiDiGraph): Input graph.

    Returns:
        graph (nx.MultiDiGraph): Updated graph with community labels and 'x' attributes.
    """
    # Convert MultiDiGraph to Graph (undirected graph for LPA)
    undirected_graph = nx.Graph(graph)

    # Perform community detection using LPA
    communities = nx.community.label_propagation_communities(undirected_graph)

    # Assign community labels to nodes and add 'x' attribute
    for community_id, community in enumerate(communities):
        for node in community:
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['x'] = [community_id]  # 'x' is a feature; wrap in a list for PyTorch Geometric compatibility

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    """
    Detect communities using LPA, update graphs with community labels, and add 'x' attribute.
    
    Parameters:
        input_dir (str): Directory containing input graphs.
        output_dir (str): Directory to save updated graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process each graph file in the input directory
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        
        # Load the graph
        graph_path = os.path.join(input_dir, graph_file)
        G = nx.read_gpickle(graph_path)

        # Detect communities using LPA and label nodes
        G = detect_and_label_communities_lpa(G)

        # Save the updated graph
        updated_graph_path = os.path.join(output_dir, graph_file)
        nx.write_gpickle(G, updated_graph_path)
        print(f"Updated graph with LPA communities and 'x' attribute saved to {updated_graph_path}")


# Example usage
if __name__ == "__main__":
    # Input directory containing graphs
    input_graph_dir = "3ed_trai_h_graphs"

    # Output directory for updated graphs
    output_graph_dir = "3ed_trai_h_graphs_commun"

    # Process graphs and add community labels using LPA
    process_graphs_with_lpa(input_graph_dir, output_graph_dir)



Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1961.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_10.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_589.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_656.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_520.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_542.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1995.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_552.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs

# convert Multigraph to hetrodata

In [26]:
import torch
from torch_geometric.data import HeteroData
import networkx as nx
import os

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph with multiple edge types to a HeteroData object.
    Preserves the 'label' field in data[rel_type].edge_label.
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    # Add node-level features
    x = []
    community_labels = []
    for node in G.nodes():
        community = G.nodes[node].get('community', -1)
        community_labels.append(community)
        x.append([community])
    data['ip'].community = torch.tensor(community_labels, dtype=torch.long)
    data['ip'].x = torch.tensor(x, dtype=torch.float)

    # Process each edge from G.
    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        rel_type = ('ip', key, 'ip')
        
        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr = []
            data[rel_type].edge_label = []  # Container for the label

        data[rel_type].edge_index.append([src, dst])
        
        # ---> FIX: Extract the label and append it <---
        # Fallback to 0 if for some reason the edge doesn't have a label
        edge_lbl = edge_attrs.get('label', 0) 
        data[rel_type].edge_label.append(edge_lbl)

        feature_vec = []
        # Append corresponding features from edge_attrs
        if key == 'network':
            for attr_name in network_features:
                feature_vec.append(edge_attrs.get(attr_name, 0.0))
        elif key == 'context':
            for attr_name in context_features:
                feature_vec.append(edge_attrs.get(attr_name, 0.0))
        elif key == 'knowledge':
            for attr_name in knowledge_features:
                feature_vec.append(edge_attrs.get(attr_name, 0.0))
                
        data[rel_type].edge_attr.append(feature_vec)

    # Convert lists to tensors.
    for rel_type in data.edge_types:
        data[rel_type].edge_index = torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(data[rel_type].edge_attr, dtype=torch.float)
        if data[rel_type].edge_label:
            # Save as a standard flat tensor
            data[rel_type].edge_label = torch.tensor(data[rel_type].edge_label, dtype=torch.long)
            
    return data

def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    """
    Converts all .gpickle graphs in a directory to HeteroData objects and saves them as .pt,
    preserving the 'label' field in data[rel_type].edge_label.
    """
    os.makedirs(output_dir, exist_ok=True)
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        graph_path = os.path.join(input_dir, graph_file)
        G = nx.read_gpickle(graph_path)
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.gpickle', '.pt'))
        torch.save(hetero_data, hetero_path)
        print(f"Saved HeteroData with labels to {hetero_path}")

if __name__ == "__main__":
    input_test_dir = "3ed_trai_h_graphs_commun"         # Input .gpickle files (with communities added)
    output_test_pt_dir = "3ed_trai_h_graphs_hetero_graphs" # Output .pt files
    process_and_save_hetero_graphs_with_label(input_test_dir, output_test_pt_dir)


Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1961.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_10.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_589.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_656.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_520.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_542.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1995.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_552.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1985.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_2000.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_2010.pt
Saved HeteroData with labels to 3ed_tra

# Test for inside of graph, no need to run it

In [27]:
#was test for inside of .pt ( no need to run)
import torch
import os

def inspect_pt_file(file_path):
    """
    Inspects the contents of a .pt file and prints its structure.

    Parameters:
        file_path (str): Path to the .pt file.
    """
    data = torch.load(file_path, weights_only=False)
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Check if it's a PyTorch Geometric HeteroData object
    if isinstance(data, dict):
        print("File contains a dictionary. Keys:")
        for key, value in data.items():
            print(f"  {key}: {type(value)}")
            if isinstance(value, torch.Tensor):
                print(f"    Tensor shape: {value.shape}")
    elif hasattr(data, 'keys') and hasattr(data, 'edge_index_dict'):
        print("File contains a HeteroData object.")
        print(f"Node types: {data.node_types}")
        for node_type in data.node_types:
            print(f"  Node type '{node_type}':")
            if 'x' in data[node_type]:
                print(f"    Node features 'x': shape {data[node_type].x.shape}")
            else:
                print("    No node features ('x') found.")
            if 'num_nodes' in data[node_type]:
                print(f"    Number of nodes: {data[node_type].num_nodes}")
        
        print(f"Edge types: {data.edge_types}")
        for edge_type in data.edge_types:
            print(f"  Edge type {edge_type}:")
            if 'edge_index' in data[edge_type]:
                print(f"    Edge index: shape {data[edge_type].edge_index.shape}")
            if 'edge_attr' in data[edge_type]:
                print(f"    Edge attributes: shape {data[edge_type].edge_attr.shape}")
    else:
        print("Unknown data format.")
    print("-" * 40)

def inspect_all_pt_files(directory):
    """
    Inspects all .pt files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .pt files.
    """
    print(f"Inspecting .pt files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".pt"):
            inspect_pt_file(os.path.join(directory, file))

# Directory containing your .pt files
input_graph_dir = "3ed_trai_h_graphs_hetero_graphs"

# Inspect all files in the directory
inspect_all_pt_files(input_graph_dir)


Inspecting .pt files in directory: 3ed_trai_h_graphs_hetero_graphs
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1959.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([216, 1])
    Number of nodes: 216
Edge types: [('ip', 'network', 'ip')]
  Edge type ('ip', 'network', 'ip'):
    Edge index: shape torch.Size([2, 199])
    Edge attributes: shape torch.Size([199, 4])
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1908.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([90, 1])
    Number of nodes: 90
Edge types: [('ip', 'network', 'ip')]
  Edge type ('ip', 'network', 'ip'):
    Edge index: shape torch.Size([2, 63])
    Edge attributes: shape torch.Size([63, 4])
----------------------------------------
I

In [28]:
#was test for inside of graph ( no need to run)
import os
import networkx as nx

def inspect_community_in_gpickle(file_path):
    """
    Inspects the presence of the 'community' attribute in a .gpickle file.

    Parameters:
        file_path (str): Path to the .gpickle file.
    """
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Load the graph
    G = nx.read_gpickle(file_path)

    # Check for 'community' attribute in nodes
    if all('community' in G.nodes[node] for node in G.nodes()):
        print(f"All nodes have a 'community' attribute.")
        print("Sample 'community' values:")
        sample_communities = {node: G.nodes[node]['community'] for node in list(G.nodes)[:10]}
        print(sample_communities)
    else:
        missing = [node for node in G.nodes() if 'community' not in G.nodes[node]]
        print(f"Some nodes are missing the 'community' attribute. Missing nodes: {missing[:10]} (only showing first 10)")

    print(f"Total nodes: {len(G.nodes())}")
    print("-" * 40)


def inspect_all_gpickle_files(directory):
    """
    Inspects the 'community' attribute in all .gpickle files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .gpickle files.
    """
    print(f"Inspecting .gpickle files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".gpickle"):
            inspect_community_in_gpickle(os.path.join(directory, file))


# Directory containing your .gpickle files
input_graph_dir = "3ed_trai_h_graphs_commun"

# Inspect all files in the directory for the 'community' attribute
inspect_all_gpickle_files(input_graph_dir)


Inspecting .gpickle files in directory: 3ed_trai_h_graphs_commun
Inspecting file: 3ed_trai_h_graphs_commun/test_graph_hour_1961.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'41.164.3.244': 0, '192.168.1.193': 0, '7.185.192.168': 1, '1.35.192.168': 1, '139.129.192.168': 2, '1.195.192.168': 2, '141.240.192.168': 2, '169.187.192.168': 1, '182.175.192.168': 3, '1.1.192.168': 3}
Total nodes: 180
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/test_graph_hour_10.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'177.30.87.144': 0, '192.168.1.1': 0, '119.221.192.168': 1, '1.152.192.168': 1, '159.236.3.122': 2, '49.24.192.168': 2, '230.113.192.168': 3, '1.152.3.122': 3, '179.177.145.115': 4, '192.168.1.193': 4}
Total nodes: 32
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/test_graph_hou